# Phase 2 — GEDI footprint download (Chapada dos Veadeiros)Downloads GEDI L4A (biomass) and L2A (quality metrics) footprints, spatiallysubsetted to the study area using NASA Harmony.**Pipeline in this notebook**| Step | What it does ||------|--------------|| 1 | Setup: config, Earth Engine, study area, native-vegetation mask || 2 | Export the study area as GeoJSON (Harmony needs it to clip) || 3 | Authenticate with NASA Earthdata || 4 | Connect to Harmony and resolve the product IDs || 5 | Define which variables and which year to request || 6 | Submit + download L4A (biomass) || 7 | Submit + download L2A (quality metrics) || 8 | Verify what was downloaded |**Why two products?** `agbd` (biomass, the target variable) is only in L4A.`num_detectedmodes` (the most effective noise filter, Moudry et al. 2024) is onlyin L2A. They are joined by `shot_number`, the unique footprint identifier.**Why Harmony?** The full L4A archive over this area is ~56 GB for 2020-2022,and almost all of it falls outside the study area. Harmony clips on NASA servers,so only the relevant subset is downloaded.Next notebook: `03_gedi_join_filter.ipynb` (read HDF5, join by shot_number, quality filter).

---## 1. SetupRun this cell first, always. If the kernel restarts, every variable is lost andthis cell rebuilds them.

In [ ]:
from datetime import datetime
from glob import glob
from pathlib import Path
import json
import os

import ee
import h5py
import earthaccess
import requests
from harmony import Client, Collection, Request

from agb_cerrado.utils import load_config
from agb_cerrado.data import load_study_area, verify_area, load_mapbiomas_mask

# Project configuration (no hardcoded values below this point)
cfg = load_config("../config/config.yaml")

ee.Initialize(project=cfg["project"]["gee_project"])
print("Earth Engine connected")

# Study area: Chapada dos Veadeiros + buffer
sa = cfg["study_area"]
aoi = load_study_area(sa["clip_region_bbox"], sa["filter_name"], sa["buffer_m"])
area_km2 = verify_area(aoi, sa["area_km2_expected"])
print(f"Study area: {area_km2:.2f} km2")

# Native-vegetation mask (MapBiomas). Used later, in the filtering notebook.
native_mask = load_mapbiomas_mask(aoi, cfg["data"]["landcover_keep_classes"], year=2021)
print("Native-vegetation mask ready")

---## 2. Export the study area as GeoJSONHarmony clips the data using a polygon file. The geometry is simplified first:the raw Earth Engine polygon contains near-duplicate vertices that Harmonyrejects (`Bad Request: The shape contained duplicate points`).`maxError=100` means vertices may shift up to 100 m — negligible for a~12,000 km2 area, and enough to remove the duplicates.

In [ ]:
GEOJSON_PATH = "../data/interim/aoi_chapada.geojson"

Path(GEOJSON_PATH).parent.mkdir(parents=True, exist_ok=True)

aoi_simplified = aoi.simplify(maxError=100)
geometry = aoi_simplified.getInfo()

feature_collection = {
    "type": "FeatureCollection",
    "features": [{"type": "Feature", "properties": {}, "geometry": geometry}],
}

with open(GEOJSON_PATH, "w") as f:
    json.dump(feature_collection, f)

n_vertices = len(geometry["coordinates"][0])
print(f"Saved: {GEOJSON_PATH}")
print(f"Vertices: {n_vertices}")

---## 3. NASA Earthdata authenticationCredentials are read from `~/.netrc`, never written in the notebook.To create the file (run once, in a terminal, with your own credentials):```bashecho "machine urs.earthdata.nasa.gov login YOUR_USER password YOUR_PASSWORD" > ~/.netrcchmod 600 ~/.netrc```

In [ ]:
auth = earthaccess.login(strategy="netrc")
print(f"Authenticated with NASA Earthdata: {auth.authenticated}")

---## 4. Harmony client and product IDsEach NASA dataset has a *concept ID*. It is resolved from the DOI so thenotebook does not depend on hardcoded identifiers that may change.- L4A — Footprint Level Aboveground Biomass Density, `10.3334/ORNLDAAC/2056`- L2A — Elevation and Height Metrics, `10.5067/GEDI/GEDI02_A.002`

In [ ]:
harmony_client = Client()


def get_concept_id(doi: str) -> str:
    """Resolve a NASA CMR concept ID from a dataset DOI."""
    url = f"https://cmr.earthdata.nasa.gov/search/collections.json?doi={doi}"
    return requests.get(url).json()["feed"]["entry"][0]["id"]


concept_l4a = get_concept_id("10.3334/ORNLDAAC/2056")
concept_l2a = get_concept_id("10.5067/GEDI/GEDI02_A.002")

collection_l4a = Collection(id=concept_l4a)
collection_l2a = Collection(id=concept_l2a)

print(f"L4A: {concept_l4a}")
print(f"L2A: {concept_l2a}")

---## 5. Request definition### Which yearStart with a single year to validate the whole pipeline end to end. Once thejoin and the filter work, change `YEAR` and re-run to add 2021 and 2022.### Which variables, and whyEvery variable requested corresponds to a filtering criterion or to a requiredpiece of information. Nothing else is requested — the products contain 100+variables, and requesting all of them would defeat the purpose of subsetting.**L4A**| Variable | Purpose ||----------|---------|| `agbd` | Target variable (aboveground biomass density) || `agbd_se` | GEDI's own uncertainty for each footprint || `l4_quality_flag` | Biomass quality filter (Kellner et al. 2023) || `l2_quality_flag` | Waveform quality filter || `degrade_flag` | Degraded positioning/pointing (Beck et al. 2021) || `sensitivity` | Beam penetration filter (Oliveira et al. 2023) || `solar_elevation` | Night-only filter (`< 0`) || `lat_lowestmode`, `lon_lowestmode` | Footprint coordinates || `shot_number` | Join key (included automatically by the subsetter) |**L2A**| Variable | Purpose ||----------|---------|| `shot_number` | Join key — must be requested explicitly here || `num_detectedmodes` | Most effective noise filter (Moudry et al. 2024) || `elev_lowestmode` | GEDI ground elevation || `digital_elevation_model` | TanDEM-X reference, for the topographic filter || `quality_flag`, `sensitivity`, `solar_elevation`, `degrade_flag` | Cross-check against L4A || `lat_lowestmode`, `lon_lowestmode` | Footprint coordinates |**Not requested:** RH (canopy height) metrics. L4A biomass is derived from them,so using them as model predictors would be data leakage.

In [ ]:
YEAR = 2020

temporal_range = {"start": datetime(YEAR, 1, 1), "stop": datetime(YEAR, 12, 31)}

BEAMS = [
    "BEAM0000", "BEAM0001", "BEAM0010", "BEAM0011",  # coverage beams
    "BEAM0101", "BEAM0110", "BEAM1000", "BEAM1011",  # full power beams
]


def per_beam(variables: list) -> list:
    """Expand variable names to the /BEAMXXXX/variable paths Harmony expects."""
    return [f"/{beam}/{var}" for beam in BEAMS for var in variables]


variables_l4a = per_beam([
    "agbd",
    "agbd_se",
    "l4_quality_flag",
    "l2_quality_flag",
    "degrade_flag",
    "sensitivity",
    "solar_elevation",
    "lat_lowestmode",
    "lon_lowestmode",
])

variables_l2a = per_beam([
    "shot_number",
    "num_detectedmodes",
    "sensitivity",
    "solar_elevation",
    "degrade_flag",
    "elev_lowestmode",
    "digital_elevation_model",
    "quality_flag",
    "lat_lowestmode",
    "lon_lowestmode",
])

DOWNLOAD_DIR = f"../data/raw/gedi_{YEAR}"
Path(DOWNLOAD_DIR).mkdir(parents=True, exist_ok=True)

print(f"Year: {YEAR}")
print(f"L4A: {len(variables_l4a)} paths ({len(variables_l4a) // 8} variables x 8 beams)")
print(f"L2A: {len(variables_l2a)} paths ({len(variables_l2a) // 8} variables x 8 beams)")
print(f"Download directory: {DOWNLOAD_DIR}")

---## 6. Submit and downloadOne function for both products. Harmony processes the request on NASA servers(this takes several minutes and shows a progress bar), then the clipped filesare downloaded.`ignore_errors=True` lets Harmony skip individual problematic granules insteadof failing the whole job. A `Job is running with errors` message is expectedand not a failure.

In [ ]:
def submit_and_download(collection, variables, product_label: str) -> list:
    """Submit a Harmony subset request and download the resulting files.

    Args:
        collection: Harmony Collection object.
        variables: List of /BEAMXXXX/variable paths to request.
        product_label: Short name used for logging, e.g. "L4A".

    Returns:
        List of paths to the downloaded *_subsetted.h5 files.
    """
    request = Request(
        collection=collection,
        variables=variables,
        temporal=temporal_range,
        shape=GEOJSON_PATH,
        ignore_errors=True,
    )

    job_id = harmony_client.submit(request)
    print(f"[{product_label}] job submitted: {job_id}")

    harmony_client.result_json(job_id, show_progress=True)
    print(f"[{product_label}] processing done, downloading...")

    futures = harmony_client.download_all(job_id, directory=DOWNLOAD_DIR, overwrite=True)
    files = [f.result() for f in futures]

    print(f"[{product_label}] {len(files)} files downloaded")
    return files

---## 7. Download L4A (biomass)

In [ ]:
files_l4a = submit_and_download(collection_l4a, variables_l4a, "L4A")

---## 8. Download L2A (quality metrics)

In [ ]:
files_l2a = submit_and_download(collection_l2a, variables_l2a, "L2A")

---## 9. Verify the downloadTwo checks before moving on:1. **File counts match.** Each L4A orbit granule should have an L2A counterpart,   otherwise some footprints cannot be joined.2. **`shot_number` is present in both products.** Without it there is no join key.

In [ ]:
l4a_on_disk = sorted(glob(os.path.join(DOWNLOAD_DIR, "*GEDI04_A*_subsetted.h5")))
l2a_on_disk = sorted(glob(os.path.join(DOWNLOAD_DIR, "*GEDI02_A*_subsetted.h5")))

print(f"L4A files: {len(l4a_on_disk)}")
print(f"L2A files: {len(l2a_on_disk)}")


def first_populated_beam(h5_path: str):
    """Return (beam_name, variable_list) for the first beam containing data."""
    with h5py.File(h5_path, "r") as f:
        for beam in [k for k in f.keys() if k.startswith("BEAM")]:
            keys = list(f[beam].keys())
            if keys:
                return beam, keys
    return None, []


for label, files in [("L4A", l4a_on_disk), ("L2A", l2a_on_disk)]:
    if not files:
        print(f"\n{label}: no files found")
        continue
    beam, variables = first_populated_beam(files[0])
    has_shot = "shot_number" in variables
    print(f"\n{label} — variables in {beam}:")
    for v in variables:
        print(f"    {v}")
    print(f"  shot_number present: {has_shot}")

---## Next steps1. **To add more years:** change `YEAR` in section 5, then re-run sections 5-9.2. **To join and filter:** continue in `03_gedi_join_filter.ipynb`.### References for the filtering criteria- Beck, J., Armston, J., Hofton, M., & Luthcke, S. (2021). *GEDI Level 02 User Guide*. USGS EROS.- Kellner, J. R., Armston, J., & Duncanson, L. (2023). Algorithm theoretical basis document for GEDI footprint aboveground biomass density. *Earth and Space Science*, 10(4).- Moudry, V., et al. (2024). How to find accurate terrain and canopy height GEDI footprints in temperate forests and grasslands? *Earth and Space Science*, 11.- Oliveira, P. V., Zhang, X., Peterson, B., & Ometto, J. P. (2023). Using simulated GEDI waveforms to evaluate the effects of beam sensitivity and terrain slope on GEDI L2A relative height metrics over the Brazilian Amazon Forest. *Science of Remote Sensing*, 7.